<a href="https://colab.research.google.com/github/KaraIbr/SmartParkingGU/blob/main/SmartParkingr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmartParking - Sistema de Deteccion de Ocupacion de Estacionamiento

## Pipeline Completo: EDA + Preprocesamiento + Entrenamiento

In [ ]:
import os
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.feature import hog
from skimage.color import rgb2gray

sns.set_style("whitegrid")
print("Librerias importadas correctamente")

## 1. Conexion con Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Busqueda Automatica del Dataset

In [ ]:
DATASET_NAME = "CNRPark-Patches-150x150"
dataset_path = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if DATASET_NAME in dirs:
        dataset_path = os.path.join(root, DATASET_NAME)
        break

if dataset_path is None:
    raise FileNotFoundError(f"No se encontro la carpeta {DATASET_NAME}")

print(f"Dataset encontrado en:\n{dataset_path}")

## 3. Deteccion de Clases

In [ ]:
subfolders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
print("Carpetas encontradas:", subfolders)

ocupado_dir = None
libre_dir = None

for folder in subfolders:
    nombre = folder.lower()
    if "ocup" in nombre or "occupied" in nombre:
        ocupado_dir = os.path.join(dataset_path, folder)
    if any(k in nombre for k in ["libre", "empty", "vacant", "no_ocupado", "free"]):
        libre_dir = os.path.join(dataset_path, folder)

if ocupado_dir is None or libre_dir is None:
    if len(subfolders) >= 2:
        ocupado_dir = os.path.join(dataset_path, subfolders[0])
        libre_dir = os.path.join(dataset_path, subfolders[1])

print(f"\nClase Ocupado: {ocupado_dir}")
print(f"Clase Libre: {libre_dir}")

## 4. Conteo de Imagenes

In [ ]:
valid_ext = (".jpg", ".jpeg", ".png", ".bmp")

n_ocupadas = len([f for f in os.listdir(ocupado_dir) if f.lower().endswith(valid_ext)])
n_libres = len([f for f in os.listdir(libre_dir) if f.lower().endswith(valid_ext)])
total = n_ocupadas + n_libres

print("=" * 40)
print("INFORMACION GENERAL DEL DATASET")
print("=" * 40)
print(f"Total de imagenes: {total}")
print(f"Ocupadas:          {n_ocupadas} ({n_ocupadas / total * 100:.1f}%)")
print(f"Libres:            {n_libres} ({n_libres / total * 100:.1f}%)")
print("=" * 40)

## 5. Grafico de Distribucion de Clases

In [ ]:
df_clases = pd.DataFrame({
    "Clase": ["Ocupado", "Libre"],
    "Cantidad": [n_ocupadas, n_libres]
})

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_clases, x="Clase", y="Cantidad", palette=["#e74c3c", "#2ecc71"])

for i, valor in enumerate(df_clases["Cantidad"]):
    porcentaje = valor / total * 100
    ax.text(i, valor + (total * 0.01), f"{valor}\n({porcentaje:.1f}%)",
            ha="center", va="bottom", fontsize=11)

plt.title("Distribucion de imagenes por categoria", fontsize=14, fontweight="bold")
plt.xlabel("Clase", fontsize=12)
plt.ylabel("Cantidad de imagenes", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Muestra Representativa de Imagenes

In [ ]:
def mostrar_muestra(directorio, titulo, n=4):
    archivos = [f for f in os.listdir(directorio) if f.lower().endswith(valid_ext)][:n]
    fig, axes = plt.subplots(1, n, figsize=(15, 4))
    fig.suptitle(titulo, fontsize=14, fontweight="bold")
    for i, archivo in enumerate(archivos):
        img = Image.open(os.path.join(directorio, archivo))
        axes[i].imshow(img)
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

mostrar_muestra(ocupado_dir, "Espacios Ocupados")
mostrar_muestra(libre_dir, "Espacios Libres")

## 7. Analisis de Resolucion

In [ ]:
def obtener_resoluciones(directorio):
    resoluciones = []
    for f in os.listdir(directorio):
        if f.lower().endswith(valid_ext):
            img = Image.open(os.path.join(directorio, f))
            resoluciones.append(img.size)
    return resoluciones

res_ocupadas = obtener_resoluciones(ocupado_dir)
res_libres = obtener_resoluciones(libre_dir)
todas_res = res_ocupadas + res_libres

anchos = [r[0] for r in todas_res]
altos = [r[1] for r in todas_res]

print("=" * 40)
print("RESOLUCION DE IMAGENES")
print("=" * 40)
print(f"Resolucion minima:  {min(anchos)}x{min(altos)}")
print(f"Resolucion maxima:  {max(anchos)}x{max(altos)}")
print(f"Resolucion promedio: {int(np.mean(anchos))}x{int(np.mean(altos))}")
print(f"Total imagenes analizadas: {len(todas_res)}")
print("=" * 40)

## 8. Division del Dataset

| Conjunto | Porcentaje |
|----------|------------|
| Entrenamiento | 70% |
| Validacion | 15% |
| Prueba | 15% |

In [ ]:
source_dir = dataset_path
output_dir = './dataset'
splits = {'train': 0.7, 'test': 0.15, 'validation': 0.15}
categories = ['ocupado', 'no_ocupado']

for split in ['entrenamiento', 'test', 'validacion']:
    for cat in categories:
        os.makedirs(os.path.join(output_dir, split, cat), exist_ok=True)

clase_dirs = {'ocupado': ocupado_dir, 'no_ocupado': libre_dir}

for cat, cat_dir in clase_dirs.items():
    if cat_dir is None or not os.path.exists(cat_dir):
        print(f"Directorio no encontrado para: {cat}")
        continue

    files = [f for f in os.listdir(cat_dir)
             if f.lower().endswith(valid_ext)]
    random.shuffle(files)

    total_cat = len(files)
    train_count = int(total_cat * splits['train'])
    test_count = int(total_cat * splits['test'])

    train_files = files[:train_count]
    test_files = files[train_count:train_count + test_count]
    val_files = files[train_count + test_count:]

    for file in train_files:
        shutil.copyfile(os.path.join(cat_dir, file),
                        os.path.join(output_dir, 'entrenamiento', cat, file))
    for file in test_files:
        shutil.copyfile(os.path.join(cat_dir, file),
                        os.path.join(output_dir, 'test', cat, file))
    for file in val_files:
        shutil.copyfile(os.path.join(cat_dir, file),
                        os.path.join(output_dir, 'validacion', cat, file))

    print(f"{cat}: total={total_cat}, train={len(train_files)}, test={len(test_files)}, val={len(val_files)}")

print(f"\nDataset organizado en: {output_dir}")

## 9. Visualizacion HOG

Pipeline: Original -> Grayscale -> HOG

In [ ]:
def visualizar_hog(directorio, titulo):
    archivos = [f for f in os.listdir(directorio) if f.lower().endswith(valid_ext)][0]
    img = Image.open(os.path.join(directorio, archivos))
    img_np = np.array(img)
    img_gray = rgb2gray(img_np)

    features, hog_image = hog(
        img_gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True,
        block_norm='L2-Hys'
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Pipeline HOG - {titulo}", fontsize=14, fontweight="bold")

    axes[0].imshow(img_np)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(img_gray, cmap="gray")
    axes[1].set_title("Escala de Grises")
    axes[1].axis("off")

    axes[2].imshow(hog_image, cmap="gray")
    axes[2].set_title("HOG")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

visualizar_hog(ocupado_dir, "Ocupado")
visualizar_hog(libre_dir, "Libre")

## 10. Data Augmentation

Cuadricula 2x2: Original, Brillo, Ruido, Contraste

In [ ]:
def aplicar_brillo(img, factor=1.5):
    img_array = np.array(img, dtype=np.float32)
    img_bright = np.clip(img_array * factor, 0, 255).astype(np.uint8)
    return Image.fromarray(img_bright)

def agregar_ruido(img, sigma=25):
    img_array = np.array(img, dtype=np.float32)
    ruido = np.random.normal(0, sigma, img_array.shape)
    img_ruido = np.clip(img_array + ruido, 0, 255).astype(np.uint8)
    return Image.fromarray(img_ruido)

def cambiar_contraste(img, factor=1.8):
    img_array = np.array(img, dtype=np.float32)
    mean = np.mean(img_array)
    img_contraste = np.clip(mean + factor * (img_array - mean), 0, 255).astype(np.uint8)
    return Image.fromarray(img_contraste)

archivos = [f for f in os.listdir(ocupado_dir) if f.lower().endswith(valid_ext)][:1]
img_original = Image.open(os.path.join(ocupado_dir, archivos[0]))

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
fig.suptitle("Data Augmentation", fontsize=14, fontweight="bold")

axes[0, 0].imshow(img_original)
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

axes[0, 1].imshow(aplicar_brillo(img_original))
axes[0, 1].set_title("Brillo (+50%)")
axes[0, 1].axis("off")

axes[1, 0].imshow(agregar_ruido(img_original))
axes[1, 0].set_title("Ruido Gaussiano")
axes[1, 0].axis("off")

axes[1, 1].imshow(cambiar_contraste(img_original))
axes[1, 1].set_title("Contraste (+80%)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 11. Conclusiones del EDA

1. El dataset presenta una distribucion equilibrada entre clases (~50% ocupado, ~50% libre).

2. Las imagenes mantienen dimensiones homogeneas despues del preprocesamiento (150x150).

3. Las caracteristicas HOG permiten resaltar bordes y formas relevantes para distinguir espacios ocupados y libres.